## **Dependencies**

In [1]:
!pip install -q -U rank-bm25 nltk tqdm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.3/57.3 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 27.1 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.2/80.2 kB 5.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dopamine-rl 4.1.2 requires gym<=0.25.2, but you have gym 0.26.2 which is incompatible.
moviepy 1.0.3 requires decorator<5.0,>=4.0.2, but you have decorator 5.3.1 which is incompatible.


In [2]:
#!pip install -q -U transformers accelerate torch

## **Imports**

In [19]:
import json
import os
import pickle
import re
from typing import Dict, List, Set, Any

import math
import numpy as np
from tqdm import tqdm
from collections import Counter

import torch
from sentence_transformers import CrossEncoder
from transformers import AutoModelForCausalLM, AutoTokenizer
from rank_bm25 import BM25Okapi

import nltk
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer

In [20]:
nltk.pathsec.ALLOW_PROXIED_FETCH = True
nltk.download('stopwords', quiet=True)

True

## **Configuration Paths**

In [21]:
DATA_DIR = "/kaggle/input/datasets/anarvaaa/original-scifact-data"
DEV_CLAIMS_PATH = os.path.join(DATA_DIR, "claims_dev.jsonl")
INDEX_PKL_PATH = os.path.join(DATA_DIR, "scifact_bm25_index.pkl")

In [22]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using compute device: {DEVICE}")
EXPANSION_MODEL_ID = "HuggingFaceTB/SmolLM2-360M-Instruct"
RERANKER_MODEL_NAME = "BAAI/bge-reranker-base"
CANDIDATE_POOL_SIZE = 50 
FINAL_TOP_K = 5

Using compute device: cuda


## **Preprocessing**

In [23]:
stop_words = set(stopwords.words("english"))
stemmer = PorterStemmer()

def tokenize(text: str, remove_stopwords: bool = True, use_stemming: bool = True) -> List[str]:
    # Clean non-alphanumeric noise to protect BM25 token matches
    tokens = re.findall(r"\b[a-zA-Z0-9]+(?:-[a-zA-Z0-9]+)*\b", text.lower())
    if remove_stopwords:
        tokens = [t for t in tokens if t not in stop_words]
    if use_stemming:
        tokens = [stemmer.stem(t) for t in tokens]
    return tokens

## **Load Index**

In [24]:
print(f"Loading BM25 index from: {INDEX_PKL_PATH}")
with open(INDEX_PKL_PATH, "rb") as f:
    bm25_artifacts = pickle.load(f)

bm25: BM25Okapi = bm25_artifacts["bm25_model"]
doc_ids: List[int] = bm25_artifacts["doc_ids"]
doc_metadata: Dict[int, Dict[str, Any]] = bm25_artifacts["doc_metadata"]

print(f"Successfully loaded index with {len(doc_ids):,} indexed documents.")

Loading BM25 index from: /kaggle/input/datasets/anarvaaa/original-scifact-data/scifact_bm25_index.pkl
Successfully loaded index with 5,183 indexed documents.


## **Load Models**

In [25]:
print(f"Loading Expansion Model: {EXPANSION_MODEL_ID}...")
tokenizer = AutoTokenizer.from_pretrained(EXPANSION_MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    EXPANSION_MODEL_ID,
    torch_dtype=torch.float16 if DEVICE == "cuda" else torch.float32,
    device_map="auto" if DEVICE == "cuda" else None
)
model.eval()

print(f"Loading Cross-Encoder Reranker: {RERANKER_MODEL_NAME}...")
reranker = CrossEncoder(RERANKER_MODEL_NAME, max_length=512, device=DEVICE)

Loading Expansion Model: HuggingFaceTB/SmolLM2-360M-Instruct...


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Loading Cross-Encoder Reranker: BAAI/bge-reranker-base...


config.json:   0%|          | 0.00/799 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.11G [00:00<?, ?B/s]

...

XLMRobertaForSequenceClassification LOAD REPORT from: BAAI/bge-reranker-base
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json:   0%|          | 0.00/443 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/279 [00:00<?, ?B/s]

## **BM25 Retrieval**

In [26]:
def retrieve_bm25(query_str: str, k: int = CANDIDATE_POOL_SIZE) -> List[Dict[str, Any]]:
    tokens = tokenize(query_str)
    scores = bm25.get_scores(tokens)
    top_indices = sorted(range(len(scores)), key=lambda i: scores[i], reverse=True)[:k]
    
    results = []
    for rank, idx in enumerate(top_indices, start=1):
        d_id = doc_ids[idx]
        results.append({
            "doc_id": d_id,
            "rank": rank,
            "score": round(float(scores[idx]), 4),
            "title": doc_metadata[d_id]["title"],
            "abstract_text": doc_metadata[d_id]["abstract_text"]
        })
    return results

## **Query Expansion**

In [27]:
def generate_kl_prf_expanded_query(
    claim: str, 
    fb_docs: int = 5, 
    fb_terms: int = 5
) -> Dict[str, Any]:
    """
    KL-Divergence Pseudo-Relevance Feedback:
    Selects expansion terms showing high information gain in top feedback documents.
    """
    initial_candidates = retrieve_bm25(claim, k=fb_docs)
    if not initial_candidates:
        return {"expanded_terms": [], "expanded_query": claim}

    claim_tokens = set(tokenize(claim, remove_stopwords=True, use_stemming=False))
    
    # Extract terms across feedback pool documents
    pool_tokens = []
    term_doc_count = Counter()
    
    for doc in initial_candidates:
        text = f"{doc['title']} {doc['abstract_text']}"
        tokens = tokenize(text, remove_stopwords=True, use_stemming=False)
        
        # Clean & filter candidate terms
        filtered = [
            t for t in tokens 
            if len(t) > 2 and not t.isdigit() and t not in claim_tokens
        ]
        
        pool_tokens.extend(filtered)
        term_doc_count.update(set(filtered))

    if not pool_tokens:
        return {"expanded_terms": [], "expanded_query": claim}

    pool_tf = Counter(pool_tokens)
    total_pool_terms = len(pool_tokens)
    total_fb_docs = len(initial_candidates)
    
    kl_scores = {}
    for term, count in pool_tf.items():
        # Probability of term in the feedback pool
        p_pool = count / total_pool_terms
        
        # Estimated background probability based on document frequency in local pool
        p_bg = term_doc_count[term] / (total_fb_docs * 100.0)
        
        # Information Gain / KL-Divergence metric
        if p_pool > p_bg:
            kl_scores[term] = p_pool * math.log2(p_pool / p_bg)

    # Pick top k terms with highest KL divergence score
    sorted_terms = sorted(kl_scores.items(), key=lambda x: x[1], reverse=True)
    top_expansion_terms = [t[0] for t in sorted_terms[:fb_terms]]
    
    expansion_str = " ".join(top_expansion_terms)
    expanded_query_str = f"{claim} {expansion_str}".strip()

    return {
        "expanded_terms": top_expansion_terms,
        "expanded_query": expanded_query_str
    }

## **Re-Ranking**

In [28]:
def rerank_documents_hybrid(
    expanded_query: str, 
    candidate_docs: List[Dict[str, Any]], 
    alpha: float = 0.7
) -> List[Dict[str, Any]]:
    """
    Hybrid Re-ranking:
    1. Evaluates candidate passages against the EXPANDED query.
    2. Combines Min-Max normalized Cross-Encoder scores with BM25 scores to prevent query drift.
    """
    if not candidate_docs:
        return []

    # 1. Match against Expanded Query
    pairs = [
        [expanded_query, f"{doc['title']} {doc['abstract_text']}".strip()] 
        for doc in candidate_docs
    ]
    
    ce_scores = reranker.predict(pairs)

    # 2. Normalize BM25 and Cross-Encoder scores for linear interpolation
    bm25_raw = [d["score"] for d in candidate_docs]
    ce_raw = list(ce_scores)

    min_bm25, max_bm25 = min(bm25_raw), max(bm25_raw)
    min_ce, max_ce = min(ce_raw), max(ce_raw)

    bm25_norm = [(s - min_bm25) / (max_bm25 - min_bm25 + 1e-6) for s in bm25_raw]
    ce_norm = [(s - min_ce) / (max_ce - min_ce + 1e-6) for s in ce_raw]

    # 3. Score Fusion: Final_Score = alpha * CE_norm + (1 - alpha) * BM25_norm
    reranked_docs = []
    for idx, doc in enumerate(candidate_docs):
        doc_copy = doc.copy()
        fusion_score = (alpha * ce_norm[idx]) + ((1 - alpha) * bm25_norm[idx])
        doc_copy["ce_score"] = float(ce_scores[idx])
        doc_copy["fusion_score"] = float(fusion_score)
        reranked_docs.append(doc_copy)

    return sorted(reranked_docs, key=lambda x: x["fusion_score"], reverse=True)

## **Test Run**

In [29]:
print(f"\nLoading sample queries from: {DEV_CLAIMS_PATH}")
sample_claims = []
with open(DEV_CLAIMS_PATH, "r", encoding="utf-8") as f:
    for line in f:
        item = json.loads(line.strip())
        sample_claims.append(item)
        if len(sample_claims) == 10:
            break

print(f"Loaded {len(sample_claims)} test claims for evaluation.\n")

for idx, claim_item in enumerate(sample_claims, start=1):
    q_id = claim_item["id"]
    query = claim_item["claim"]
    gold_doc_ids = claim_item.get("cited_doc_ids", [])
    
    print("=" * 100)
    print(f"[{idx}/10] CLAIM ID {q_id}: \"{query}\"")
    print(f"Gold Cited Doc IDs: {gold_doc_ids}")
    print("-" * 100)

    # Stage 1: Original BM25 Retrieval
    orig_bm25_docs = retrieve_bm25(query, k=CANDIDATE_POOL_SIZE)
    orig_top_5_ids = [d["doc_id"] for d in orig_bm25_docs[:FINAL_TOP_K]]
    orig_hits = [d_id for d_id in gold_doc_ids if d_id in orig_top_5_ids]

    # Stage 2: PRF Expansion + BM25 Pass 2
    prf_data = generate_kl_prf_expanded_query(query, fb_docs=5, fb_terms=5)
    expanded_query = prf_data["expanded_query"]
    print(f"PRF Extracted Terms: {prf_data['expanded_terms']}")
    print(f"Expanded Query: \"{expanded_query}\"")

    exp_bm25_docs = retrieve_bm25(expanded_query, k=CANDIDATE_POOL_SIZE)
    exp_top_5_ids = [d["doc_id"] for d in exp_bm25_docs[:FINAL_TOP_K]]
    exp_hits = [d_id for d_id in gold_doc_ids if d_id in exp_top_5_ids]

    # Candidate Pool Recall Check
    candidate_ids = [d["doc_id"] for d in exp_bm25_docs]
    in_pool = [d_id for d_id in gold_doc_ids if d_id in candidate_ids]
    print(f"BM25 Candidate Pool Recall (Top {CANDIDATE_POOL_SIZE}): {in_pool} / {gold_doc_ids}")

    # Stage 3: Cross-Encoder Hybrid Re-ranking (evaluating against expanded_query with score fusion)
    reranked_docs = rerank_documents_hybrid(expanded_query, exp_bm25_docs, alpha=0.7)
    reranked_top_5_ids = [d["doc_id"] for d in reranked_docs[:FINAL_TOP_K]]
    reranked_hits = [d_id for d_id in gold_doc_ids if d_id in reranked_top_5_ids]

    # Metrics Summary
    print(f"\nGold Hits Summary:")
    print(f" - Original BM25 (Top 5):   {orig_hits} / {gold_doc_ids}")
    print(f" - PRF BM25 (Top 5):        {exp_hits} / {gold_doc_ids}")
    print(f" - CE Re-ranked (Top 5):    {reranked_hits} / {gold_doc_ids}")

    # 3-Stage Tracking Table
    print("\n   3-Stage Document Ranking Tracking (Top 5):")
    print(f"   {'Rank':<5} | {'Original BM25':<28} | {'PRF Expanded BM25':<28} | {'Re-ranked (Hybrid Fusion)':<28}")
    print("   " + "-" * 92)
    for r in range(FINAL_TOP_K):
        orig_str = f"{orig_bm25_docs[r]['doc_id']} ({orig_bm25_docs[r]['score']:.2f})" if r < len(orig_bm25_docs) else "N/A"
        exp_str  = f"{exp_bm25_docs[r]['doc_id']} ({exp_bm25_docs[r]['score']:.2f})" if r < len(exp_bm25_docs) else "N/A"
        rr_str   = f"{reranked_docs[r]['doc_id']} ({reranked_docs[r]['fusion_score']:.4f})" if r < len(reranked_docs) else "N/A"
        print(f"   {r+1:<5} | {orig_str:<28} | {exp_str:<28} | {rr_str:<28}")


Loading sample queries from: /kaggle/input/datasets/anarvaaa/original-scifact-data/claims_dev.jsonl
Loaded 10 test claims for evaluation.

[1/10] CLAIM ID 1: "0-dimensional biomaterials show inductive properties."
Gold Cited Doc IDs: [31715818]
----------------------------------------------------------------------------------------------------
PRF Extracted Terms: ['cells', 'ltp', 'escs', 'human', 'stem']
Expanded Query: "0-dimensional biomaterials show inductive properties. cells ltp escs human stem"
BM25 Candidate Pool Recall (Top 50): [] / [31715818]

Gold Hits Summary:
 - Original BM25 (Top 5):   [] / [31715818]
 - PRF BM25 (Top 5):        [] / [31715818]
 - CE Re-ranked (Top 5):    [] / [31715818]

   3-Stage Document Ranking Tracking (Top 5):
   Rank  | Original BM25                | PRF Expanded BM25            | Re-ranked (Hybrid Fusion)   
   --------------------------------------------------------------------------------------------
   1     | 18953920 (9.89)              | 

## **Ablation Style Results**

In [30]:
print(f"\nLoading dev queries from: {DEV_CLAIMS_PATH}")
all_claims = []
with open(DEV_CLAIMS_PATH, "r", encoding="utf-8") as f:
    for line in f:
        item = json.loads(line.strip())
        all_claims.append(item)

print(f"Loaded {len(all_claims)} total claims for evaluation processing.\n")

evaluation_records = []

for idx, claim_item in enumerate(tqdm(all_claims, desc="Processing Claims"), start=1):
    q_id = claim_item["id"]
    query = claim_item["claim"]
    gold_doc_ids = claim_item.get("cited_doc_ids", [])
    gold_evidence = claim_item.get("evidence", {})

    # ---------------------------------------------------------
    # 1. Base BM25 Pass (Original Query)
    # ---------------------------------------------------------
    orig_bm25_docs = retrieve_bm25(query, k=CANDIDATE_POOL_SIZE)

    # Config 1: BM25 Only
    bm25_only_results = [
        {"doc_id": d["doc_id"], "score": d["score"], "rank": i + 1}
        for i, d in enumerate(orig_bm25_docs)
    ]

    # ---------------------------------------------------------
    # 2. PRF Query Expansion
    # ---------------------------------------------------------
    prf_data = generate_kl_prf_expanded_query(query, fb_docs=5, fb_terms=5)
    expanded_query = prf_data["expanded_query"]
    extracted_terms = prf_data["expanded_terms"]

    # ---------------------------------------------------------
    # 3. Expansion BM25 Pass (Expanded Query)
    # ---------------------------------------------------------
    exp_bm25_docs = retrieve_bm25(expanded_query, k=CANDIDATE_POOL_SIZE)

    # Config 2: Expansion Only
    expansion_only_results = [
        {"doc_id": d["doc_id"], "score": d["score"], "rank": i + 1}
        for i, d in enumerate(exp_bm25_docs)
    ]

    # ---------------------------------------------------------
    # 4. Config 3: Re-ranking Only (CE on Original BM25 Pool)
    # ---------------------------------------------------------
    rerank_orig_docs = rerank_documents_hybrid(query, orig_bm25_docs, alpha=0.7)
    reranking_only_results = [
        {
            "doc_id": d["doc_id"],
            "fusion_score": d["fusion_score"],
            "ce_score": d["ce_score"],
            "rank": i + 1
        }
        for i, d in enumerate(rerank_orig_docs)
    ]

    # ---------------------------------------------------------
    # 5. Config 4: Expansion + Re-ranking (CE on Expanded BM25 Pool)
    # ---------------------------------------------------------
    rerank_exp_docs = rerank_documents_hybrid(expanded_query, exp_bm25_docs, alpha=0.7)
    expansion_plus_reranking_results = [
        {
            "doc_id": d["doc_id"],
            "fusion_score": d["fusion_score"],
            "ce_score": d["ce_score"],
            "rank": i + 1
        }
        for i, d in enumerate(rerank_exp_docs)
    ]

    # Structure complete claim record
    record = {
        "claim_id": q_id,
        "claim_text": query,
        "gold_doc_ids": gold_doc_ids,
        "gold_evidence": gold_evidence,
        "prf_expanded_query": expanded_query,
        "prf_extracted_terms": extracted_terms,
        "runs": {
            "bm25_only": bm25_only_results,
            "expansion_only": expansion_only_results,
            "reranking_only": reranking_only_results,
            "expansion_plus_reranking": expansion_plus_reranking_results
        }
    }
    
    evaluation_records.append(record)

print(f"\nSuccessfully generated evaluation results for {len(evaluation_records)} claims.")


Loading dev queries from: /kaggle/input/datasets/anarvaaa/original-scifact-data/claims_dev.jsonl
Loaded 300 total claims for evaluation processing.



Processing Claims: 100%|██████████| 300/300 [17:09<00:00,  3.43s/it]


Successfully generated evaluation results for 300 claims.


In [31]:
OUTPUT_EVAL_FILE = "/kaggle/working/scifact_retrieval_eval_results.json"

with open(OUTPUT_EVAL_FILE, "w", encoding="utf-8") as f:
    json.dump(evaluation_records, f, indent=2)

print(f"Evaluation dataset saved successfully to: {OUTPUT_EVAL_FILE}")
print(f"File size: {os.path.getsize(OUTPUT_EVAL_FILE) / (1024 * 1024):.2f} MB")

Evaluation dataset saved successfully to: /kaggle/working/scifact_retrieval_eval_results.json
File size: 7.72 MB
